### Elastic Net Logistic Regression Baseline_Lasso_Ridge
#### Nicholas Perry

In [1]:
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from tabulate import tabulate
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score,RocCurveDisplay,confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
import os
#print(os.getcwd())

In [4]:
#Read in data
e_cigarette_data = pd.read_csv("cleaned_data.csv")
e_cigarette_data = e_cigarette_data.dropna(subset=['E_Cig_User'])
e_cigarette_data = e_cigarette_data.drop(columns=['Unnamed: 0'])
x = e_cigarette_data.drop('E_Cig_User', axis = 1)
x_train_scaled = pd.read_csv("x_train.csv")
x_test_scaled = pd.read_csv("x_test.csv")
y_train_scaled = pd.read_csv("y_train.csv")
y_test_scaled = pd.read_csv("y_test.csv")

In [ ]:
x_train_scaled = x_train_scaled.drop(columns=['Age_Category_Elderly','Age_Category_Young_Adult','Education_College_4_years_or_more_(college_graduate)',
                                             'Has_Disability','Marital_Status_Married'])

In [ ]:
x_test_scaled = x_test_scaled.drop(columns=['Age_Category_Elderly','Age_Category_Young_Adult','Education_College_4_years_or_more_(college_graduate)',
                                             'Has_Disability','Marital_Status_Married'])

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
# Create a pipeline to scale features and apply Elastic Net Logistic Regression
pipeline = Pipeline([  # Standardize features
    ('model', LogisticRegression(
        penalty='elasticnet',
        solver='saga',  # Required for Elastic Net
        l1_ratio=0.5,  # If set to 1, exclusively lasso. If set to 0, exclusively ridge. 0.5 balances the two
        #Setting to 0.5 to create a baseline logistic regression
        max_iter=100  # Increase iterations for convergence
    ))
])

# Fit the model
pipeline.fit(x_train_scaled, y_train_scaled)

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
# Make predictions
y_pred = pipeline.predict(x_test_scaled)
y_prob = pipeline.predict_proba(x_test_scaled)[:, 1]

# Evaluate model performance
print("Confusion Matrix:",confusion_matrix(y_test_scaled, y_pred))
print("Precision:", precision_score(y_test_scaled, y_pred))
print("Recall:", recall_score(y_test_scaled, y_pred))
print("F1 Score:", f1_score(y_test_scaled, y_pred))
print("AUC-ROC:", roc_auc_score(y_test_scaled, y_prob))
RocCurveDisplay.from_estimator(pipeline,x_test_scaled, y_test_scaled)
plt.show()
cm = confusion_matrix(y_test_scaled, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix = cm)
disp.plot()
plt.show()

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
#Perform a grid search with CV to find coefficients for variables to be removed
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='roc_auc')
grid_search.fit(x_train_scaled, y_train_scaled)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

In [ ]:
x

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
#These are the variables to be removed
# Get feature coefficients
coefficients = grid_search.best_estimator_.named_steps['model'].coef_[0]
feature_importance = pd.DataFrame({
    'Feature': x_train_scaled.columns,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(feature_importance.head())

Ridge

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
# Create a pipeline to scale features and apply Elastic Net Logistic Regression
###CUSTOM FUNCTION MODIFIED FROM https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
pipeline = Pipeline([  # Standardize features
    ('model', LogisticRegression(
        penalty='elasticnet',
        solver='saga',  # Required for Elastic Net
        l1_ratio=0,  # If set to 1, exclusively lasso. If set to 0, exclusively ridge. 0.5 balances the two
        #Setting to 0.5 to create a baseline logistic regression
        max_iter=100  # Increase iterations for convergence
    ))
])

# Fit the model
pipeline.fit(x_train_scaled, y_train_scaled)

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
# Make predictions
y_pred = pipeline.predict(x_test_scaled)
y_prob = pipeline.predict_proba(x_test_scaled)[:, 1]

# Evaluate model performance
print("Confusion Matrix:",confusion_matrix(y_test_scaled, y_pred))
print("Precision:", precision_score(y_test_scaled, y_pred))
print("Recall:", recall_score(y_test_scaled, y_pred))
print("F1 Score:", f1_score(y_test_scaled, y_pred))
print("AUC-ROC:", roc_auc_score(y_test_scaled, y_prob))
RocCurveDisplay.from_estimator(pipeline,x_test_scaled, y_test_scaled)
plt.show()
cm = confusion_matrix(y_test_scaled, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix = cm)
disp.plot()
plt.show()

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
#Perform a grid search with CV to find coefficients for variables to be removed
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='roc_auc')
grid_search.fit(x_train_scaled, y_train_scaled)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
#These are the variables to be removed
# Get feature coefficients
coefficients = grid_search.best_estimator_.named_steps['model'].coef_[0]
feature_importance = pd.DataFrame({
    'Feature': x_train_scaled.columns,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(feature_importance.head())

Lasso

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
# Create a pipeline to scale features and apply Elastic Net Logistic Regression
###CUSTOM FUNCTION MODIFIED FROM https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
pipeline = Pipeline([  # Standardize features
    ('model', LogisticRegression(
        penalty='elasticnet',
        solver='saga',  # Required for Elastic Net
        l1_ratio=1,  # If set to 1, exclusively lasso. If set to 0, exclusively ridge. 0.5 balances the two
        #Setting to 0.5 to create a baseline logistic regression
        max_iter=100  # Increase iterations for convergence
    ))
])

# Fit the model
pipeline.fit(x_train_scaled, y_train_scaled)

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
# Make predictions
y_pred = pipeline.predict(x_test_scaled)
y_prob = pipeline.predict_proba(x_test_scaled)[:, 1]

# Evaluate model performance
print("Confusion Matrix:",confusion_matrix(y_test_scaled, y_pred))
print("Precision:", precision_score(y_test_scaled, y_pred))
print("Recall:", recall_score(y_test_scaled, y_pred))
print("F1 Score:", f1_score(y_test_scaled, y_pred))
print("AUC-ROC:", roc_auc_score(y_test_scaled, y_prob))
RocCurveDisplay.from_estimator(pipeline,x_test_scaled, y_test_scaled)
plt.show()
cm = confusion_matrix(y_test_scaled, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix = cm)
disp.plot()
plt.show()

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
#Perform a grid search with CV to find coefficients for variables to be removed

param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='roc_auc')
grid_search.fit(x_train_scaled, y_train_scaled)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

In [ ]:
#Code Modified from https://ujangriswanto08.medium.com/building-robust-models-with-elastic-net-logistic-regression-in-python-68bdfa381cc5
#These are the variables to be removed
# Get feature coefficients
coefficients = grid_search.best_estimator_.named_steps['model'].coef_[0]
feature_importance = pd.DataFrame({
    'Feature': x_train_scaled.columns,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(feature_importance.head())